In [1]:
import pandas as pd 
from sklearn.metrics import roc_auc_score

from risk_calculator import RiskCalculator

In [2]:
df=pd.read_csv("../data/merged/multi_turn_data.csv")

In [3]:
def recompute_risks(df, calc):
    df = df.sort_values(["conv_id", "turn_id"]).copy()

    interaction_list  = []
    pattern_list      = []
    progressive_list  = []

    for conv_id, group in df.groupby("conv_id"):
        prev = 0.0
        for _, row in group.iterrows():
            interaction = calc.compute_interaction_risk(row)
            pattern     = calc.compute_pattern_risk(row)
            prog        = calc.calculate_progressive_risk(row, prev)
            prev        = prog
            interaction_list.append(interaction)
            pattern_list.append(pattern)
            progressive_list.append(prog)

    df["interaction_risk"] = interaction_list
    df["pattern_risk"]     = pattern_list
    df["progressive_risk"] = progressive_list

 
    prev_prog_list = []
    for conv_id, group in df.groupby("conv_id"):
        progs = df.loc[group.index, "progressive_risk"].tolist()
        prev_prog_list.extend([0.0] + progs[:-1])
    df["prev_progressive"] = prev_prog_list

    return df


In [ ]:
def suggest_triplet(trial, prefix):

    a = trial.suggest_float(f"{prefix}_a", 0, 1)
    b = trial.suggest_float(f"{prefix}_b", 0, 1-a)
    c = 1 - a - b

    return a,b,c

In [12]:
def objective(trial):

    alpha,beta,gamma = suggest_triplet(trial, "main")

    ia,ib,ig = suggest_triplet(trial, "inter")

    pa,pb,pg = suggest_triplet(trial, "pattern")

    params = {
        "alpha":alpha,
        "beta":beta,
        "gamma":gamma,

        "inter_alpha":ia,
        "inter_beta":ib,
        "inter_gamma":ig,

        "pattern_alpha":pa,
        "pattern_beta":pb,
        "pattern_gamma":pg,
    }

    risk_calc = RiskCalculator(**params)

    df_new = recompute_risks(df.copy(), risk_calc)

    score = roc_auc_score(
        df_new["label"],
        df_new["progressive_risk"]
    )

    return score

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=1000)

[I 2026-06-06 16:06:29,672] A new study created in memory with name: no-name-544b3ba8-256e-4810-97e4-e4dc9b9798c6
[I 2026-06-06 16:06:29,924] Trial 0 finished with value: 0.41748022386796835 and parameters: {'main_a': 0.02009301076387804, 'main_b': 0.31596375020331585, 'inter_a': 0.9837834145528064, 'inter_b': 0.008269188750030729, 'pattern_a': 0.4003275268851244, 'pattern_b': 0.16138791318190224}. Best is trial 0 with value: 0.41748022386796835.
[I 2026-06-06 16:06:30,154] Trial 1 finished with value: 0.45721350633154156 and parameters: {'main_a': 0.28023203774384076, 'main_b': 0.24803921736465415, 'inter_a': 0.9752681607591098, 'inter_b': 0.021839741966764463, 'pattern_a': 0.579256607198464, 'pattern_b': 0.35683854353575944}. Best is trial 1 with value: 0.45721350633154156.
[I 2026-06-06 16:06:30,384] Trial 2 finished with value: 0.5254164192406663 and parameters: {'main_a': 0.2016045331758145, 'main_b': 0.7299246832197058, 'inter_a': 0.6967985571633876, 'inter_b': 0.2801433945134297

In [13]:
print(study.best_value)
print(study.best_params)

0.6189424217903078
{'main_a': 0.19893217044930248, 'main_b': 0.7850219386375682, 'inter_a': 0.9998914527596705, 'inter_b': 2.9042989394090715e-05, 'pattern_a': 0.04928947684122276, 'pattern_b': 0.9262176628719436}


In [16]:
import json
import numpy as np

bp = study.best_params

alpha = float(bp["main_a"])
beta  = float(bp["main_b"])
gamma = 1 - alpha - beta

inter_alpha = float(bp["inter_a"])
inter_beta  = float(bp["inter_b"])
inter_gamma = 1 - inter_alpha - inter_beta

pattern_alpha = float(bp["pattern_a"])
pattern_beta  = float(bp["pattern_b"])
pattern_gamma = 1 - pattern_alpha - pattern_beta

final_params = {
    "alpha": alpha,
    "beta": beta,
    "gamma": gamma,

    "inter_alpha": inter_alpha,
    "inter_beta": inter_beta,
    "inter_gamma": inter_gamma,

    "pattern_alpha": pattern_alpha,
    "pattern_beta": pattern_beta,
    "pattern_gamma": pattern_gamma,
}

with open("../config/optimized_params_risk.json", "w") as f:
    json.dump(final_params, f, indent=4)